# LlamaIndex와 AgentCore 메모리 - 의료 지식 어시스턴트 (장기 메모리)

## 소개

이 노트북은 Amazon Bedrock AgentCore 메모리 기능을 LlamaIndex와 통합하여 **장기 메모리**를 활용한 의료 지식 어시스턴트를 만드는 방법을 보여줍니다. 여러 환자 상담과 의료 사례에 걸쳐 지속적으로 의료 지식을 축적하고, 수개월에서 수년에 걸친 환자 케어를 추적할 수 있습니다.

## 아키텍처 개요

![LlamaIndex AgentCore 장기 메모리 아키텍처](LlamaIndex-AgentCore-LTM-Arch.png)

## 튜토리얼 세부 정보

**튜토리얼 세부 정보:**
- **튜토리얼 유형**: 장기 크로스 세션 메모리
- **에이전트 사용 사례**: 의료 지식 어시스턴트
- **에이전트 프레임워크**: LlamaIndex
- **LLM 모델**: Anthropic Claude 3.7 Sonnet
- **튜토리얼 구성 요소**: AgentCore 장기 메모리, LlamaIndex 에이전트, 의료 도구
- **예제 복잡도**: 고급

## 비즈니스 가치

**기업 의료 인텔리전스**: 환자 지식을 축적하고, 치료 진화를 추적하며, 사례와 기간에 걸쳐 포괄적인 의료 이력을 유지하는 지속적 AI 메모리로 헬스케어 업무를 혁신하세요.

**주요 전문적 이점:**
- **환자 연속성**: 진료 만남 및 케어 팀 간 원활한 지식 이전
- **임상 기억**: 중요한 환자 이력, 치료 및 결과를 영구적으로 보존
- **크로스 환자 인텔리전스**: 여러 환자 사례에 걸친 패턴과 연관성 파악
- **치료 우수성**: 과거 환자 데이터를 활용한 우수한 케어 의사결정
- **인구 건강**: 장기 환자 케어를 위한 상세한 컨텍스트 유지
- **품질 개선**: 치료 프로토콜과 그 효과를 시간에 걸쳐 추적

## 장기 메모리 구성

**기술 설정**: 이 튜토리얼은 12개월 보존을 위한 시맨틱 전략과 함께 AgentCore 메모리를 사용합니다:
- **메모리 유형**: 자동 인사이트 추출을 위한 시맨틱 전략
- **보존 기간**: 환자 케어 연속성을 위한 365일 이벤트 만료
- **크로스 세션**: 동일 actor_id + memory_id, 의료 기간별 다른 session_id
- **검색 기능**: 환자 이력 전반에 걸친 시맨틱 검색을 위한 내장 메모리 검색 도구

## 기술 개요

**주요 장기 메모리 구성 요소:**
1. **시맨틱 전략 구성**: 365일 보존의 자동 인사이트 추출을 위한 SemanticStrategy 사용
2. **크로스 세션 지속성**: 동일 actor_id + memory_id, 기간별 다른 session_id로 지식 연속성 확보
3. **커스텀 메모리 검색 도구**: AgentCore의 네이티브 search_long_term_memories()를 LlamaIndex FunctionTool로 래핑
4. **시맨틱 처리 파이프라인**: 대화 이벤트 → 시맨틱 메모리 변환을 위한 120초 대기
5. **동적 세션 관리**: 유연한 세션 처리를 위한 memory.context.session_id 사용

**학습 내용:**

- 여러 환자 상담에 걸친 지속적 AgentCore 메모리 생성
- 시간에 따른 누적 의료 지식 구축
- 환자 이력 및 치료 결과 전반에 걸친 시맨틱 검색 구현
- 치료 효과 및 의료 인사이트 진화 추적
- 크로스 세션 의료 지식 지속성 및 검색 테스트

## 시나리오 컨텍스트

이 예제에서는 수개월에서 수년에 걸친 여러 환자 상담에서 의료 지식을 유지하는 "의료 지식 어시스턴트"를 만듭니다. 어시스턴트는 AgentCore 메모리를 사용하여 환자 이력, 치료 프로토콜, 약물 상호작용, 임상 결과에 대한 지속적 지식 베이스를 구축하며, 이는 시간이 지남에 따라 성장하고 진화하여 정교한 종합적 의료 분석을 가능하게 합니다.

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델 접근 권한

## 1단계: 종속성 설치 및 설정

In [ ]:
# 시맨틱 전략 툴킷을 포함한 필수 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3 bedrock-agentcore-starter-toolkit

In [ ]:
# 필수 컴포넌트 임포트
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies.semantic import SemanticStrategy
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

print("모든 종속성을 성공적으로 임포트했습니다!")

## 2단계: AgentCore 메모리 구성

장기 의료 지식을 위한 AgentCore 메모리 리소스를 생성하거나 가져옵니다:

In [ ]:
# 장기 지속성을 위한 시맨틱 전략으로 AgentCore 메모리 생성
region = os.getenv('AWS_REGION', 'us-east-1')
memory_manager = MemoryManager(region_name=region)

try:
    # 자동 인사이트 추출을 위한 시맨틱 전략으로 메모리 생성
    memory = memory_manager.get_or_create_memory(
        name=f'MedicalAssistantSemantic_{int(datetime.now().timestamp())}',
        strategies=[SemanticStrategy(name="medicalLongTermMemory")],
        event_expiry_days=365  # 의료 기록을 위한 12개월 보존
    )
    memory_id = memory.get('id')
    print(f"시맨틱 메모리 생성 완료: {memory_id}")
    print(f"   상태: {memory.get('status')}")
    print(f"   전략: {[s.get('name') if isinstance(s, dict) else str(s) for s in memory.get('strategies', [])]}")
    
    # 메모리가 ACTIVE 상태가 될 때까지 대기
    if memory.get('status') != 'ACTIVE':
        print(f"\n메모리가 ACTIVE 상태가 될 때까지 대기 중 (현재 {memory.get('status')})...")
        import time
        max_wait = 300  # 최대 5분
        waited = 0
        while waited < max_wait:
            time.sleep(10)
            waited += 10
            # 상태 확인
            current_memory = memory_manager.get_memory(memory_id)
            status = current_memory.get('status')
            print(f"   [{waited}초] 상태: {status}")
            if status == 'ACTIVE':
                print(f"메모리가 ACTIVE 상태가 되었습니다! ({waited}초 소요)")
                break
        else:
            print(f"경고: {max_wait}초 후에도 메모리가 ACTIVE 상태가 아닙니다. 계속 진행합니다...")
    
except Exception as e:
    print(f"메모리 생성 오류: {e}")
    memory_id = "your-memory-id-here"  # 기존 메모리 ID로 교체하세요

## 3단계: 의료 도구 구현

종합적 의료 분석을 위한 전문 도구를 정의합니다:

In [ ]:
def assess_patient_symptoms(patient_id: str, symptoms: str, severity: str, duration: str) -> str:
    """Assess patient symptoms with severity and duration tracking"""
    return f"환자 증상 평가 완료: {patient_id} (중증도: {severity}, 기간: {duration})"

def check_drug_interactions(patient_id: str, medications: str, interaction_level: str, recommendations: str) -> str:
    """Check drug interactions with safety recommendations"""
    return f"약물 상호작용 확인: {patient_id} - {interaction_level} - {recommendations}"

def document_treatment_protocol(protocol_type: str, indication: str, effectiveness: str, side_effects: str) -> str:
    """Document treatment protocol with effectiveness and side effects"""
    print(f"치료 프로토콜: {protocol_type} - {indication} (효과: {effectiveness})")
    return f"치료 프로토콜 기록 완료: {protocol_type}"

def update_clinical_guideline(patient_id: str, guideline_type: str, recommendation: str, evidence_level: str) -> str:
    """Update clinical guideline for specific patient"""
    print(f"임상 가이드라인: {patient_id} - {guideline_type} ({evidence_level} 근거 수준)")
    return f"가이드라인 업데이트 완료: {patient_id}"

def log_treatment_outcome(patient_id: str, treatment: str, outcome: str, follow_up_needed: str) -> str:
    """Log treatment outcome with follow-up requirements"""
    print(f"치료 결과: {patient_id} - {treatment}: {outcome}")
    return f"결과 기록 완료: {patient_id}"

def log_medical_milestone(quarter: str, milestone: str, details: str) -> str:
    """Log a medical milestone with quarter and detailed progress"""
    print(f"의료 마일스톤 - {quarter}: {milestone}")
    return f"{quarter} 마일스톤 기록 완료: {milestone} - {details}"

def track_clinical_metrics(metric_type: str, value: str, patient_id: str, quarter: str) -> str:
    """Track specific clinical metrics with patient and timeline"""
    print(f"{quarter}: {metric_type} = {value} ({patient_id} 대상)")
    return f"{metric_type} 추적 완료: {patient_id}의 {quarter} - {value}"

def save_medical_insight(insight: str, quarter: str, clinical_context: str) -> str:
    """Save medical insights with clinical context"""
    print(f"{quarter} 인사이트: {insight[:50]}...")
    return f"{quarter} 인사이트 저장 완료 (임상 컨텍스트: {clinical_context})"

# 에이전트를 위한 도구 객체 생성
medical_tools = [
    FunctionTool.from_defaults(fn=assess_patient_symptoms),
    FunctionTool.from_defaults(fn=check_drug_interactions),
    FunctionTool.from_defaults(fn=document_treatment_protocol),
    FunctionTool.from_defaults(fn=update_clinical_guideline),
    FunctionTool.from_defaults(fn=log_treatment_outcome),
    FunctionTool.from_defaults(fn=log_medical_milestone),
    FunctionTool.from_defaults(fn=track_clinical_metrics),
    FunctionTool.from_defaults(fn=save_medical_insight)
]

print("의료 도구 생성 완료!")

## 3b단계: 메모리 검색 도구 추가

에이전트가 장기 메모리를 검색할 수 있는 도구를 생성합니다:

In [ ]:
def create_memory_retrieval_tool(memory_id: str, actor_id: str, region: str):
    """Create a tool for the agent to search its own long-term memory"""
    
    def search_long_term_memory(query: str) -> str:
        """Search long-term memory for relevant medical information about patients, treatments, protocols, and outcomes.
        
        Use this tool when you need to recall:
        - Patient information (symptoms, treatments, outcomes)
        - Treatment protocols and their effectiveness
        - Drug interactions and safety profiles
        - Clinical guidelines and recommendations
        - Medical insights and lessons learned
        
        Args:
            query: Search query describing what information you need (e.g., 'PATIENT-001 symptoms', 'diabetes protocols', 'Q1 outcomes')
        
        Returns:
            Relevant information from long-term memory
        """
        try:
            from bedrock_agentcore.memory.session import MemorySessionManager
            
            # 세션 매니저 생성 (memory_id와 region만 필요)
            session_manager = MemorySessionManager(
                memory_id=memory_id,
                region_name=region
            )
            
            # 시맨틱 전략 네임스페이스에서 장기 메모리 검색
            results = session_manager.search_long_term_memories(
                query=query,
                namespace_prefix="/strategies/",  # 시맨틱 전략 네임스페이스에서 검색
                top_k=5,
                max_results=10
            )
            
            if not results:
                return "장기 메모리에서 관련 정보를 찾지 못했습니다. 새로운 정보이거나 메모리 추출이 아직 처리 중일 수 있습니다."
            
            # 에이전트를 위한 결과 포맷팅
            output = "장기 메모리에서 검색된 정보:\\n\\n"
            for i, result in enumerate(results, 1):
                # MemoryRecord 객체 - content 속성에 접근
                content = getattr(result, 'content', str(result))
                # 매우 긴 콘텐츠 잘라내기
                if len(content) > 300:
                    content = content[:300] + "..."
                output += f"{i}. {content}\\n\\n"
            
            return output
            
        except Exception as e:
            return f"메모리 검색 오류: {str(e)}. 과거 컨텍스트 없이 진행합니다."
    
    return FunctionTool.from_defaults(fn=search_long_term_memory)

# 메모리 검색 도구 생성
memory_search_tool = create_memory_retrieval_tool(memory_id, "medical-assistant", region)

# 도구 목록에 메모리 검색 추가
medical_tools_with_memory = medical_tools + [memory_search_tool]

print(f"메모리 검색 도구 생성 완료! 총 도구 수: {len(medical_tools_with_memory)}")
print("   사용 네임스페이스: /strategies/ (시맨틱 전략 호환용)")

## 3c단계: 메모리 구성 확인

시맨틱 전략이 올바르게 구성되었는지 확인합니다:

In [ ]:
# 메모리 구성 확인
memory_info = memory_manager.get_memory(memory_id)
print(f"전략: {memory_info.get('strategies')}")
print(f"상태: {memory_info.get('status')}")
print(f"이름: {memory_info.get('name')}")

# 전략 세부 정보 표시
strategies = memory_info.get('strategies', [])
for strategy in strategies:
    print(f"\n전략 세부 정보:")
    print(f"  이름: {strategy.get('name')}")
    print(f"  유형: {strategy.get('type')}")
    print(f"  상태: {strategy.get('status')}")
    print(f"  ID: {strategy.get('strategyId')}")

## 3c단계: 메모리 구성 확인

시맨틱 전략이 올바르게 구성되었는지 확인합니다:

In [ ]:
# 메모리 구성 확인
memory_info = memory_manager.get_memory(memory_id)
print(f"전략: {memory_info.get('strategies')}")
print(f"상태: {memory_info.get('status')}")
print(f"이름: {memory_info.get('name')}")

# 전략 세부 정보 표시
strategies = memory_info.get('strategies', [])
for strategy in strategies:
    print(f"\n전략 세부 정보:")
    print(f"  이름: {strategy.get('name')}")
    print(f"  유형: {strategy.get('type')}")
    print(f"  상태: {strategy.get('status')}")
    print(f"  ID: {strategy.get('strategyId')}")

## 4단계: 멀티 세션 에이전트 구현

다양한 의료 기간을 시뮬레이션하기 위한 헬퍼 함수를 생성합니다:

In [ ]:
# 장기 메모리(크로스 세션) 구성
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
ASSISTANT_ID = "medical-assistant"  # 모든 세션에서 동일한 어시스턴트

def create_medical_session(session_name: str):
    """장기 메모리 지속성을 갖춘 새로운 의료 세션 생성"""
    context = AgentCoreMemoryContext(
        actor_id=ASSISTANT_ID,         # 동일한 어시스턴트
        memory_id=memory_id,         # 동일한 메모리 저장소 (장기 메모리 활성화)
        session_id=f"medical-{session_name}", # 기간별 다른 세션
        namespace="/medical-analysis/"
    )
    
    memory = AgentCoreMemory(context=context)
    llm = BedrockConverse(model=MODEL_ID)
    agent = FunctionAgent(
        tools=medical_tools_with_memory,  # 메모리 검색 기능이 포함된 도구 사용
        llm=llm, 
        verbose=True,  # 메모리 검색 시점을 확인하기 위해 verbose 활성화
        system_prompt="""You are a senior medical assistant with access to long-term memory.
        
CRITICAL: When asked about patients, treatments, protocols, or historical information, 
you MUST use the search_long_term_memory tool FIRST before responding.

For example:
- "What patients am I treating?" → Use search_long_term_memory("patients treatments")
- "What protocols have I used?" → Use search_long_term_memory("medical protocols")
- "What outcomes have I achieved?" → Use search_long_term_memory("patient outcomes")

Always provide conclusive, complete responses without asking follow-up questions.\n
Execute all requested actions immediately and completely. Provide detailed, professional responses."""
    )
    
    return agent, memory

print("멀티 세션 의료 지식 어시스턴트 설정 완료!")

## 5단계: Q1 의료 세션 - 초기 환자 평가

첫 번째 의료 세션을 시작하고 환자 기준선을 설정합니다:

In [ ]:
# === Q1 의료 세션 ===
print("=== Q1: 초기 환자 평가 ===")

agent_q1, memory_q1 = create_medical_session("q1")

# 초기 환자 증상 평가
# (시니어 의료 어시스턴트 Dr. Maria Rodriguez로서, PATIENT-001의 증상 평가 요청: 증상 '피로, 갈증 증가, 빈뇨, 시야 흐림', 중증도 '보통', 기간 '3주')
response = await agent_q1.run(
    "I'm Senior Medical Assistant Dr. Maria Rodriguez. Assess patient symptoms for 'PATIENT-001' with symptoms 'fatigue, increased thirst, frequent urination, blurred vision', "
    "severity 'Moderate', duration '3 weeks'.",
    memory=memory_q1
)

print("Q1 초기 평가:")
print(response)

In [ ]:
# 초기 임상 가이드라인 기록
# (PATIENT-001의 임상 가이드라인 업데이트: 가이드라인 유형 '당뇨 관리', 권고사항 '메트포르민 500mg 1일 2회 시작, 생활습관 상담, 혈당 모니터링', 높은 근거 수준)
response = await agent_q1.run(
    "Update clinical guideline for 'PATIENT-001': guideline type 'Diabetes Management', "
    "recommendation 'initiate metformin 500mg BID, lifestyle counseling, glucose monitoring', evidence level 'high'.",
    memory=memory_q1
)
print("Q1 임상 가이드라인:", response)

# (PATIENT-001의 임상 가이드라인 업데이트: 가이드라인 유형 '생활습관 수정', 권고사항 '식이 상담, 운동 프로그램, 체중 10% 감량 목표', 높은 근거 수준)
response = await agent_q1.run(
    "Update clinical guideline for 'PATIENT-001': guideline type 'Lifestyle Modification', "
    "recommendation 'dietary consultation, exercise program, weight management target 10% reduction', evidence level 'high'.",
    memory=memory_q1
)
print("Q1 생활습관 가이드라인:", response)
# 의료 발견 사항을 명시적으로 추적
# (의료 발견 사항 저장: '환자가 치료 프로토콜에 잘 반응함', 확신도 '높음')
await agent_q1.run(
    "Save medical finding: finding 'Patient responds well to treatment protocol', confidence 'high'.",
    memory=memory_q1
)

In [ ]:
# 시맨틱 메모리 처리를 위한 대기 시간
import asyncio
print("\n의료 메모리 추출 대기 중...")
await asyncio.sleep(120)
print("의료 메모리 처리 완료")

In [ ]:
# 이벤트가 저장되었는지 확인
print("\n이벤트 저장 확인 중...")
try:
    client = MemoryClient(region_name=region)
    events = client.list_events(
        memory_id=memory_id,
        actor_id="medical-assistant",  # 도메인별 ID로 대체 예정
        session_id=memory_q1.context.session_id  # 동적 세션 ID
    )
    print(f"세션에 {len(events)}개의 대화 이벤트가 저장되었습니다")
except Exception as e:
    print(f"경고: 이벤트 확인 불가: {e}")

# 시맨틱 메모리 처리를 위한 대기 시간
import asyncio
print("\n시맨틱 메모리 추출 및 인덱싱 대기 중...")
print("   (AgentCore가 백그라운드에서 대화 이벤트를 처리합니다)")
await asyncio.sleep(120)  # 메모리 추출을 위한 대기 시간 증가
print("메모리 처리 완료 - 이제 메모리를 검색할 수 있습니다")

## 6단계: Q2 의료 세션 - 치료 프로토콜 업데이트

장기 메모리 검색을 테스트하고 치료 반응에 적응합니다:

In [ ]:
# === Q2 의료 세션 ===
print("\n=== Q2: 치료 프로토콜 업데이트 (새 세션) ===")

agent_q2, memory_q2 = create_medical_session("q2")

# 크로스 세션 환자 회상 테스트 - 에이전트가 search_long_term_memory 도구를 사용해야 함
print("\n세션 간 메모리 검색 테스트 중...")
print("   (에이전트가 search_long_term_memory 도구를 사용하는지 확인하세요)\n")

# (치료 중인 환자가 누구인지, 증상, 치료, 임상 가이드라인은 무엇인지 질문)
response = await agent_q2.run(
    "What patients am I treating? What are their symptoms, treatments, and clinical guidelines?",
    memory=memory_q2
)

print("\nQ2 환자 회상:")
print(response)
print("\n예상 결과: PATIENT-001, 당뇨 증상, 메트포르민 치료, 생활습관 가이드라인")

In [ ]:
# 치료 프로토콜 기록
# (치료 프로토콜 기록: 프로토콜 유형 '제2형 당뇨 관리', 적응증 '새로 진단된 T2DM - HbA1c 8.2%', 효과 '3개월 후 HbA1c 7.1%로 감소', 부작용 '초기 경미한 위장 불편감, 식사와 함께 복용 시 해결')
response = await agent_q2.run(
    "Document treatment protocol: protocol type 'Type 2 Diabetes Management', "
    "indication 'newly diagnosed T2DM with HbA1c 8.2%', "
    "effectiveness 'HbA1c reduced to 7.1% after 3 months', "
    "side effects 'mild GI upset initially, resolved with food intake'.",
    memory=memory_q2
)
print("Q2 치료 프로토콜:", response)

# 반응에 따른 임상 가이드라인 업데이트
# (PATIENT-001의 임상 가이드라인 업데이트: 가이드라인 유형 '약물 조정', 권고사항 '메트포르민 500mg BID 유지, 추가 혈당 조절을 위해 글리피지드 5mg 일 1회 추가', 높은 근거 수준)
response = await agent_q2.run(
    "Update clinical guideline for 'PATIENT-001': guideline type 'Medication Adjustment', "
    "recommendation 'continue metformin 500mg BID, add glipizide 5mg daily for additional glycemic control', evidence level 'high'.",
    memory=memory_q2
)
print("Q2 가이드라인 업데이트:", response)

In [ ]:
# Q2 임상 지표 추적
# (PATIENT-001의 임상 지표 추적: 지표 유형 'HbA1c 개선', 값 '8.2%에서 7.1%', Q2 2024)
response = await agent_q2.run(
    "Track clinical metrics for 'PATIENT-001': metric type 'HbA1c Improvement', value '8.2% to 7.1%', "
    "patient_id 'PATIENT-001', quarter 'Q2 2024'.",
    memory=memory_q2
)
print("Q2 임상 지표:", response)

# 치료 비교 테스트
# (PATIENT-001의 Q1에서 Q2까지 치료가 어떻게 진행되었는지, 초기 vs 현재 접근 방식 비교 질문)
response = await agent_q2.run(
    "How did PATIENT-001's treatment progress from Q1 to Q2? Compare initial vs current approach.",
    memory=memory_q2
)
print("Q2 치료 분석:")
print(response)
print("\n예상 결과: Q1 메트포르민 단독요법 → Q2 병용요법, HbA1c 개선")

## 7단계: Q3 의료 세션 - 합병증 관리

합병증 관리 및 새 환자 접수로 진행합니다:

In [ ]:
# === Q3 의료 세션 ===
print("\n=== Q3: 합병증 관리 및 새 환자 ===")

agent_q3, memory_q3 = create_medical_session("q3")

# 치료 결과 기록
# (PATIENT-001의 치료 결과 기록: 치료 '당뇨 관리 프로토콜', 결과 '목표 HbA1c 달성 (6.8%), 체중 15파운드 감량, 혈압 조절됨', 추적 관찰 필요 '분기별 HbA1c 모니터링, 연간 안과 검사, 현재 처방 유지')
response = await agent_q3.run(
    "Log treatment outcome for 'PATIENT-001' with treatment 'Diabetes Management Protocol', "
    "outcome 'Target HbA1c achieved (6.8%), weight loss 15 lbs, BP controlled', "
    "follow_up_needed 'quarterly HbA1c monitoring, annual eye exam, continue current regimen'.",
    memory=memory_q3
)
print("Q3 치료 결과:", response)

# 새 환자 평가 시작
# (PATIENT-002의 증상 평가: 증상 '흉통, 호흡 곤란, 심계항진', 중증도 '높음', 기간 '2일')
response = await agent_q3.run(
    "Assess patient symptoms for 'PATIENT-002': symptoms 'chest pain, shortness of breath, palpitations', "
    "severity 'High', duration '2 days'.",
    memory=memory_q3
)
print("Q3 새 환자 평가:", response)

In [ ]:
# 포괄적인 의료 이력 회상 테스트
# (전체 의료 케어 이력 요청: 모든 환자, 치료, 프로토콜, 결과 포함)
response = await agent_q3.run(
    "What is the complete medical care history? Include all patients, treatments, "
    "protocols, and outcomes.",
    memory=memory_q3
)
print("Q3 전체 이력:")
print(response)
print("\n예상 결과: PATIENT-001 당뇨 여정 → PATIENT-002 심장 평가, 프로토콜 진화")

## 8단계: Q4 의료 세션 - 연말 리뷰 및 계획

시맨틱 검색과 연간 의료 분석을 테스트합니다:

In [ ]:
# === Q4 의료 세션 ===
print("\n=== Q4: 연말 리뷰 및 계획 ===")

agent_q4, memory_q4 = create_medical_session("q4")

# 연간 의료 지표 추적
# (임상 지표 추적: 지표 유형 '2024 연간 성과', 값 '치료 환자: 2명, 당뇨 조절 달성: 100%, 예방된 합병증: 1건', ANNUAL-SUMMARY, 2024 연간)
response = await agent_q4.run(
    "Track clinical metrics: metric type '2024 Annual Performance', value 'Patients treated: 2, Diabetes control achieved: 100%, Complications prevented: 1', "
    "patient_id 'ANNUAL-SUMMARY', quarter '2024 Annual'.",
    memory=memory_q4
)
print("Q4 연간 지표:", response)

# 치료 프로토콜 상관관계 테스트
# (올해 기록한 치료 프로토콜이 무엇이었는지, 얼마나 효과적이었는지 질문)
response = await agent_q4.run(
    "What treatment protocols have I documented this year? How effective were they?",
    memory=memory_q4
)
print("Q4 프로토콜 효과 분석:")
print(response)
print("\n예상 결과: 당뇨 관리 프로토콜 → 성공적 HbA1c 조절")

In [ ]:
# 유사한 의료 접근 방식에 대한 시맨틱 검색 테스트
# (사용한 임상 가이드라인이 무엇이었는지, 환자 결과에 기반하여 가장 효과적이었던 것은 무엇인지 질문)
response = await agent_q4.run(
    "What clinical guidelines have I used? Which were most effective based on patient outcomes?",
    memory=memory_q4
)
print("Q4 가이드라인 효과 분석:")
print(response)
print("\n예상 결과: 당뇨 관리 + 생활습관 수정 = 성공적 결과")

## 9단계: 2년차 Q1 세션 - 다년 의료 관점

장기 의료 지식과 업무 진화를 테스트합니다:

In [ ]:
# === 2년차 Q1 의료 세션 ===
print("\n=== 2년차 Q1: 다년 의료 관점 ===")

agent_y2q1, memory_y2q1 = create_medical_session("year2-q1")

# 다년 의료 업무 분석
# (의료 업무 진화 분석 요청: 지난 1년간 환자와 치료가 어떻게 발전했는지, 주요 임상 결정과 그 결과는 무엇이었는지)
response = await agent_y2q1.run(
    "Analyze my medical practice evolution: How have my patients and treatments developed over the past year? "
    "What were the key clinical decisions and their outcomes?",
    memory=memory_y2q1
)
print("2년차 Q1 업무 분석:")
print(response)
print("\n예상 결과: PATIENT-001 → PATIENT-002 진행, 프로토콜 정교화, 결과 개선")

In [ ]:
# 임상 프로토콜 진화 추적 테스트
# (임상 프로토콜과 가이드라인이 어떻게 진화했는지, 어떤 치료를 적용했고 그 이유는 무엇인지 질문)
response = await agent_y2q1.run(
    "How have my clinical protocols and guidelines evolved? What treatments have I applied and why?",
    memory=memory_y2q1
)
print("2년차 Q1 프로토콜 진화:")
print(response)
print("\n예상 결과: 당뇨 프로토콜로 시작 → 심장 케어로 확대")

## 10단계: 최종 의료 업무 평가

장기 의료 분석 역량에 대한 포괄적 테스트:

In [ ]:
# 최종 포괄적 의료 업무 쿼리
# (전체 의료 업무 포트폴리오 제공 요청: 모든 환자의 임상 여정, 치료 효과, 프로토콜 적용, 가이드라인 활용, 환자 결과 포함. 교훈과 개발된 베스트 프랙티스도 포함)
response = await agent_y2q1.run(
    "Provide my complete medical practice portfolio: all patients with their clinical journeys, "
    "treatment effectiveness, protocol applications, guideline utilization, and patient outcomes. "
    "Include lessons learned and best practices developed.",
    memory=memory_y2q1
)
print("전체 의료 업무 포트폴리오:")
print(response)
print("\n예상 결과: 전체 환자 포트폴리오 - 치료 진화, 프로토콜 효과, 결과 분석 포함")

## 자동화된 테스트 검증
메모리 통합이 올바르게 작동하는지 검증하기 위해 다음 셀을 실행하세요:

In [ ]:
# 검증 함수 인라인 정의
class TestValidator:
    def __init__(self):
        self.results = {}
    
    def validate_memory_recall(self, response):
        """에이전트가 이전에 논의된 정보를 회상할 수 있는지 확인"""
        has_content = len(response) > 50
        has_memory_indicators = any(word in response.lower() for word in 
            ['earlier', 'mentioned', 'discussed', 'previously', 'you', 'we', 'our'])
        return "통과" if (has_content and has_memory_indicators) else "실패"
    
    def validate_session_memory(self, response):
        """에이전트가 세션 내 컨텍스트를 유지하는지 확인"""
        has_memory_content = len(response) > 100 and any(word in response.lower() for word in 
            ['previous', 'earlier', 'mentioned', 'discussed', 'before', 'already'])
        return "통과" if has_memory_content else "실패"
    
    def validate_cross_reference(self, response):
        """에이전트가 현재 쿼리를 이전 컨텍스트와 연결할 수 있는지 확인"""
        connecting_words = ['relate', 'connection', 'previous', 'earlier', 'discussed', 
                           'mentioned', 'context', 'based on', 'as we', 'as i']
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "통과" if (has_connection and has_substance) else "실패"
    
    def run_validation_summary(self, test_results):
        print("종합 테스트 검증 요약")
        print("=" * 60)
        
        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "통과" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0
        
        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")
        
        print("=" * 60)
        print(f"전체 통과율: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")
        
        if pass_rate >= 80:
            print("우수: 메모리 통합이 올바르게 작동하고 있습니다!")
        elif pass_rate >= 60:
            print("양호: 대부분의 메모리 기능이 작동하지만, 일부 조사가 필요합니다")
        else:
            print("주의 필요: 메모리 통합에 심각한 문제가 있습니다")
        
        return pass_rate

validator = TestValidator()
print("검증 함수 로드 완료!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상 - 에이전트가 논의된 내용을 회상할 수 있는지 확인
# (이 세션에서 지금까지 무엇을 논의했는지 질문)
response1 = await agent_y2q1.run("What have we discussed so far in this session?", memory=memory_y2q1)
test_results['메모리 회상'] = validator.validate_memory_recall(str(response1))
print(f"응답 1 길이: {len(str(response1))} 문자")

# 테스트 2: 세션 메모리 - 에이전트가 컨텍스트를 유지하는지 확인
# (이전에 무엇에 대해 이야기했는지 질문)
response2 = await agent_y2q1.run("What did we talk about earlier?", memory=memory_y2q1)
test_results['세션 메모리'] = validator.validate_session_memory(str(response2))
print(f"응답 2 길이: {len(str(response2))} 문자")

# 테스트 3: 교차 참조 능력 - 이전 컨텍스트와 연결할 수 있는지 확인
# (이것이 이전에 논의한 내용과 어떻게 관련되는지 질문)
response3 = await agent_y2q1.run("How does this relate to what we discussed before?", memory=memory_y2q1)
test_results['교차 참조'] = validator.validate_cross_reference(str(response3))
print(f"응답 3 길이: {len(str(response3))} 문자")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 노트북에서 다음을 시연했습니다:

**장기 메모리 통합**: 크로스 세션 의료 분석을 위한 AgentCore 메모리와 LlamaIndex 활용

**환자 케어 추적**: 여러 분기에 걸친 환자 진화 및 치료 개발

**임상 인텔리전스**: 치료 프로토콜과 그 적용의 시맨틱 검색

**의료 프로토콜 진화**: 초기 평가에서 근거 기반 접근 방식으로의 자연스러운 발전

**치료 효과**: 임상 결정과 환자 결과에 대한 상세 추적

**의료 업무 우수성**: 시간에 걸친 포괄적 환자 관리 및 케어 최적화

의료 지식 어시스턴트는 장기 메모리가 어떻게 어시스턴트를 시간이 지남에 따라 더 현명해지는 지속적 의료 파트너로 변환하는지 보여줍니다. 완전한 환자 이력을 유지하고 장기 케어 기간 전반에 걸쳐 정교한 임상 지식 검색을 가능하게 합니다.

## 정리

이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제합니다:

**참고**: 메모리를 영구적으로 삭제하려는 경우에만 실행하세요. memory_id 변수에는 이 노트북에서 앞서 생성한 메모리의 ID가 포함되어 있어야 합니다.

In [ ]:
# AgentCore 메모리 리소스 정리
try:
    from bedrock_agentcore.memory import MemoryClient
    
    client = MemoryClient(region_name=region)
    client.delete_memory(memory_id)
    print(f"메모리 삭제 완료: {memory_id}")
    
except NameError as e:
    print(f"경고: 변수가 정의되지 않았습니다: {e}")
    print("노트북을 처음부터 실행하거나 변수를 수동으로 설정하세요:")
    print("# memory_id = 'your-memory-id-here'")
    print("# region = 'us-east-1'")
except Exception as e:
    print(f"메모리 삭제 오류: {e}")